# 05 - Ensemble and class weights for macro-F1

| Experiment | What is tuned | Reference |
|---|---|---|
| `ensemble` | weights of the averaged model probabilities | best single model |
| `scaling` | per-class multipliers on probabilities of the best single model | best single model |
| `ensemble+scaling` | both | best single model |

- Inputs: `val_proba_*.csv` / `test_proba_*.csv` from 02-04 (attach their notebook outputs as Kaggle input).
- Honest scores: parameters are chosen on one half of the validation split and scored on the other half (2-fold, both ways).
- Final parameters for test are chosen on the whole validation split.

## Setup

In [ ]:
import subprocess
import sys

REPO = "https://github.com/mohinska/ai_hw3_AIGenTextDetection.git"
REPO_DIR = "/tmp/repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
import itertools
import json
import os
import re

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

import common
from common import LABELS, LABEL_COL, SEED

## Config

In [ ]:
MODELS = None                         # None = every model with both val and test probabilities
WEIGHT_STEP = 0.1                     # ensemble weight grid step (weights sum to 1)
SCALE_GRID = np.round(np.arange(0.5, 2.01, 0.1), 2)   # multipliers for p_human and p_fusion (p_ai = 1)
JOINT_WEIGHT_STEP, JOINT_SCALE_GRID = 0.25, np.round(np.arange(0.5, 2.01, 0.25), 2)  # coarser joint grid
SUBMISSION_PATH = os.path.join(common.WORK_DIR, "submission_ensemble.csv")

## Load probabilities

In [ ]:
train_df, test_df, sample_sub = common.load_data()
_, val = common.split_train_val(train_df)
y_val = val[LABEL_COL].to_numpy()

val_files = {re.sub(r"^val_proba_|\.csv$", "", os.path.basename(p)): p for p in common.find_outputs("val_proba_*.csv")}
test_files = {re.sub(r"^test_proba_|\.csv$", "", os.path.basename(p)): p for p in common.find_outputs("test_proba_*.csv")}
names = sorted(set(val_files) & set(test_files)) if MODELS is None else MODELS
assert names, "no probability files found: attach outputs of 02-04 as Kaggle input"

P_val = {n: common.load_proba(val_files[n], val["id"]) for n in names}
P_test = {n: common.load_proba(test_files[n], test_df["id"]) for n in names}


def macro(y, proba):
    return f1_score(y, np.array(LABELS)[proba.argmax(1)], average="macro")


single = pd.Series({n: macro(y_val, p) for n, p in P_val.items()}, name="val_macro_f1").sort_values(ascending=False)
REF = single.index[0]
single.round(4).to_frame()

## Cross-fitted tuning

In [ ]:
def simplex(n, step):
    """All weight vectors of length n with the given step that sum to 1."""
    k = round(1 / step)
    return [np.array(c) / k for c in itertools.product(range(k + 1), repeat=n) if sum(c) == k]


def scale_vec(s_h, s_f):
    return np.array([{"human": s_h, "fusion": s_f, "ai": 1.0}[l] for l in LABELS])


def crossfit(make_proba, grid):
    """Pick params on one half of val, score on the other half (both ways); then pick final params on all val."""
    oof = np.empty(len(y_val), dtype=object)
    for a, b in StratifiedKFold(2, shuffle=True, random_state=SEED).split(np.zeros(len(y_val)), y_val):
        best = max(grid, key=lambda g: macro(y_val[a], make_proba(g, a)))
        oof[b] = np.array(LABELS)[make_proba(best, b).argmax(1)]
    final = max(grid, key=lambda g: macro(y_val, make_proba(g, slice(None))))
    return f1_score(y_val, oof, average="macro"), final


def ensemble(P, w, idx):
    return sum(wi * P[n][idx] for wi, n in zip(w, names))

In [ ]:
experiments = {
    "ensemble": (lambda w, idx: ensemble(P_val, w, idx), simplex(len(names), WEIGHT_STEP)),
    "scaling": (lambda s, idx: P_val[REF][idx] * scale_vec(*s), list(itertools.product(SCALE_GRID, SCALE_GRID))),
    "ensemble+scaling": (lambda g, idx: ensemble(P_val, g[0], idx) * scale_vec(*g[1]),
                         list(itertools.product(simplex(len(names), JOINT_WEIGHT_STEP),
                                                itertools.product(JOINT_SCALE_GRID, JOINT_SCALE_GRID)))),
}
rows, params = [{"experiment": f"reference ({REF})", "macro_f1": single[REF]}], {}
for name, (make_proba, grid) in experiments.items():
    score, params[name] = crossfit(make_proba, grid)
    rows.append({"experiment": name, "macro_f1": score, "grid_size": len(grid)})
results = pd.DataFrame(rows)
results["delta_vs_ref"] = results["macro_f1"] - single[REF]
results.round(4)

In [ ]:
# chosen parameters (fitted on the whole validation split)
def describe(name, p):
    if name == "ensemble":
        return dict(zip(names, np.round(p, 2)))
    if name == "scaling":
        return {"model": REF, "scale_human": p[0], "scale_fusion": p[1]}
    return {**dict(zip(names, np.round(p[0], 2))), "scale_human": p[1][0], "scale_fusion": p[1][1]}


chosen = {n: describe(n, p) for n, p in params.items()}
with open(os.path.join(common.WORK_DIR, "outputs", "ensemble_params.json"), "w") as f:
    json.dump(chosen, f, indent=1, default=float)
pd.DataFrame(chosen).T

## Test predictions for the best experiment

In [ ]:
BEST = results.iloc[1:].sort_values("macro_f1", ascending=False).experiment.iloc[0]
p = params[BEST]
all_idx = slice(None)
if BEST == "ensemble":
    test_proba = ensemble(P_test, p, all_idx)
elif BEST == "scaling":
    test_proba = P_test[REF] * scale_vec(*p)
else:
    test_proba = ensemble(P_test, p[0], all_idx) * scale_vec(*p[1])

submission = common.save_submission(test_df["id"], np.array(LABELS)[test_proba.argmax(1)], sample_sub, SUBMISSION_PATH)
print("best:", BEST, "| saved", SUBMISSION_PATH, submission.shape)
submission["label"].value_counts(normalize=True).round(3)

## Conclusions

| Finding | Decision |
|---|---|
| <!-- TODO --> | <!-- TODO --> |
| <!-- TODO --> | <!-- TODO --> |